# Advanced Kubeflow Pipelines: Advanced Workflows

### What you will learn
- How to implement conditional execution (if/else branching) in KFP pipelines.
- Using loops and parallel execution patterns with ParallelFor.
- Designing fan-out/fan-in patterns for distributed processing.

### Why this matters in MLOps
Real-world ML pipelines rarely follow a straight line. Models may need different preprocessing paths based on data type, hyperparameter tuning requires parallel evaluation, and ensemble methods need fan-out/fan-in coordination. Mastering these patterns is essential for building robust, production-grade pipelines.

### You're done when...
- A pipeline with conditional branching executes correctly.
- A ParallelFor loop processes multiple configurations in parallel.
- A fan-out/fan-in pattern is implemented and understood.

## The MLOps Perspective
Advanced workflow patterns separate entry-level pipelines from production-grade ones. Conditional logic enables intelligent branching (e.g., skip retraining if model performance is acceptable). Parallel loops power hyperparameter tuning at scale. Fan-out/fan-in patterns enable distributed feature engineering and ensemble model evaluation — all critical for efficient ML operations.

## Setup and Imports

In [ ]:
# Install requirements
!pip install -r requirements.txt

In [ ]:
# Import required libraries
import kfp
from kfp import dsl
from kfp.dsl import component, Condition, ParallelFor
import json

## Conditional Execution
Conditional execution allows pipelines to make runtime decisions using `kfp.dsl.Condition()`. This is essential for branching logic such as model selection based on evaluation metrics.

In [ ]:
# Define components for conditional pipeline
@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def evaluate_model(model_path: str) -> float:
    """Evaluate model and return accuracy score."""
    import numpy as np
    accuracy = np.random.uniform(0.7, 0.99)
    print(f"Model accuracy: {accuracy:.4f}")
    return float(accuracy)


@component(
    base_image="python:3.11"
)
def deploy_to_production(model_path: str) -> str:
    """Deploy model to production endpoint."""
    endpoint = f"https://api.example.com/models/prod/{model_path.split('/')[-1]}"
    print(f"Deploying model to: {endpoint}")
    return endpoint


@component(
    base_image="python:3.11"
)
def retrain_model(model_path: str, reason: str) -> str:
    """Trigger retraining process."""
    new_model_path = f"models/retrained_{reason}"
    print(f"Retraining triggered. New model: {new_model_path}")
    return new_model_path

In [ ]:
# Define a pipeline with conditional branching
@dsl.pipeline(
    name="conditional-deployment-pipeline",
    description="Pipeline that conditionally deploys or retrains based on model accuracy"
)
def conditional_deployment_pipeline(model_path: str, accuracy_threshold: float = 0.85):
    eval_task = evaluate_model(model_path=model_path)

    with Condition(eval_task.output >= accuracy_threshold, name="deploy-branch"):
        deploy_task = deploy_to_production(model_path=model_path)
        print(f"Deploying model with accuracy {eval_task.output}")

    with Condition(eval_task.output < accuracy_threshold, name="retrain-branch"):
        retrain_task = retrain_model(
            model_path=model_path,
            reason=f"accuracy_{eval_task.output:.2f}_below_threshold"
        )
        print(f"Retraining model with accuracy {eval_task.output}")

## Parallel Loops with ParallelFor
Parallel execution of repeated tasks is a key pattern for hyperparameter tuning, cross-validation, and batch processing. KFP provides `kfp.dsl.ParallelFor` for this purpose.

In [ ]:
# Define a hyperparameter tuning component
@component(
    base_image="python:3.11",
    packages_to_install=["numpy", "scikit-learn"]
)
def train_with_params(n_estimators: int, max_depth: int, learning_rate: float) -> dict:
    """Train a model with specified hyperparameters."""
    import numpy as np
    accuracy = np.random.uniform(0.75, 0.98)
    result = {
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "accuracy": float(accuracy)
    }
    print(f"Training result: {result}")
    return result


@component(
    base_image="python:3.11"
)
def select_best_model(results: list) -> dict:
    """Select the best model from parallel training runs."""
    best = max(results, key=lambda x: x.get("accuracy", 0))
    print(f"Best model: {best}")
    return best

In [ ]:
# Define a pipeline with parallel hyperparameter tuning
@dsl.pipeline(
    name="parallel-tuning-pipeline",
    description="Hyperparameter tuning with parallel execution"
)
def parallel_tuning_pipeline():
    param_grid = [
        {"n_estimators": 50, "max_depth": 5, "learning_rate": 0.01},
        {"n_estimators": 100, "max_depth": 10, "learning_rate": 0.01},
        {"n_estimators": 200, "max_depth": 15, "learning_rate": 0.001},
        {"n_estimators": 100, "max_depth": 20, "learning_rate": 0.1},
    ]
    training_tasks = []

    for params in param_grid:
        task = train_with_params(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"]
        )
        training_tasks.append(task)

    select_best_model(results=training_tasks)

## Fan-Out/Fan-In Patterns
The fan-out/fan-in pattern distributes work across multiple parallel workers (fan-out) and then aggregates the results (fan-in). This is useful for ensemble models, distributed feature engineering, and parallel data validation.

In [ ]:
# Define components for fan-out/fan-in pattern
@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def split_data(data_path: str, num_splits: int) -> list:
    """Split data into chunks for parallel processing."""
    import numpy as np
    chunks = [f"{data_path}/chunk_{i}" for i in range(num_splits)]
    print(f"Split data into {num_splits} chunks: {chunks}")
    return chunks


@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def process_chunk(chunk_path: str) -> dict:
    """Process a single data chunk."""
    import numpy as np
    statistics = {
        "chunk": chunk_path,
        "mean": float(np.random.uniform(10, 50)),
        "std": float(np.random.uniform(1, 10)),
        "count": int(np.random.randint(100, 1000))
    }
    print(f"Processed chunk: {statistics}")
    return statistics


@component(
    base_image="python:3.11"
)
def aggregate_results(results: list) -> dict:
    """Aggregate results from all parallel chunks."""
    total_count = sum(r.get("count", 0) for r in results)
    avg_mean = sum(r.get("mean", 0) for r in results) / len(results)
    aggregated = {
        "total_records": total_count,
        "overall_mean": avg_mean,
        "num_chunks": len(results),
        "chunks": [r["chunk"] for r in results]
    }
    print(f"Aggregated result: {aggregated}")
    return aggregated

In [ ]:
# Define a fan-out/fan-in pipeline
@dsl.pipeline(
    name="fan-out-fan-in-pipeline",
    description="Demonstrates fan-out parallel processing with fan-in aggregation"
)
def fan_out_fan_in_pipeline(data_path: str = "/data/latest"):
    num_splits = 4
    split_task = split_data(data_path=data_path, num_splits=num_splits)

    with ParallelFor(items=split_task.output) as chunk:
        process_chunk(chunk_path=chunk)

    # Collect all chunk results via fan-in
    chunk_results = [
        process_chunk(chunk_path=f"{data_path}/chunk_{i}")
        for i in range(num_splits)
    ]
    aggregate_results(results=chunk_results)

## Fill-in-the-Blank Exercises

In [ ]:
# Exercise 1: Complete the condition to check if model accuracy is above 0.90
@dsl.pipeline(name="exercise-conditional")
def exercise_conditional_pipeline(accuracy: float = ___):
    with ___(accuracy >= 0.90, name="high-accuracy"):
        print("Model meets accuracy threshold")
    with dsl.Condition(accuracy < 0.90, name="low-accuracy"):
        print("Model needs improvement")
# Hint: Fill in the default parameter and the Condition class

In [ ]:
# Exercise 2: Complete the ParallelFor loop over hyperparameter combinations
hyperparams = [{"lr": 0.001, "batch_size": 32}, {"lr": 0.01, "batch_size": 64}]

# @dsl.pipeline(name="exercise-parallel")
# def exercise_parallel_pipeline():
#     with ___(items=___) as params:
#         print(f"Training with lr={params['lr']}, batch_size={params['batch_size']}")

# Hint: Use ParallelFor and pass the hyperparams list

## Summary
In this notebook, you learned:
- Conditional execution with `dsl.Condition()` for intelligent branching
- Parallel execution loops with `dsl.ParallelFor`
- Fan-out/fan-in patterns for distributed processing and aggregation
- These patterns enable sophisticated, production-grade ML pipelines